# RAG: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

**Document:** *CATAN — Game Rules & Almanac* (5th English edition, Catan Studio, 16 pages)
**LLM:** Google Gemini Flash (via `google-genai`; the exact model per answer is printed in Section 6.2)
**Bi-Encoder:** `sentence-transformers/all-MiniLM-L6-v2` | **Cross-Encoder:** `cross-encoder/ms-marco-MiniLM-L-6-v2`

Why this document? The Catan rulebook is split into short **Game Rules** (pp. 2–5) and an alphabetical **Almanac** (pp. 6–14).
A rule is often stated in one place and then *qualified* somewhere else ("except…", "you may not…", "not even 3:1").
That makes it a good stress test for retrieval: the chunk that looks most similar is frequently not the chunk that decides the answer.

**Notebook outline**
1. Setup & configuration
2. Load and clean the PDF
3. Chunking
4. Pipeline 1 — Bi-Encoder retrieval (Top-10 → Top-3 → Answer 1)
5. Pipeline 2 — Cross-Encoder re-ranking (Top-10 → re-rank → Top-3 → Answer 2)
6. Failure-mode analysis (rank shifts + Answer 1 vs. Answer 2)
7. Wider evaluation over a pool of rule questions
8. Latency profiling
9. Scaling measurement (why not cross-encode everything?)

## 1. Setup & configuration

In [1]:
# Standard library
import os, re, json, time, hashlib, statistics, textwrap, urllib.request
from pathlib import Path

# Third-party
import fitz                                  # PyMuPDF: PDF text extraction with font/position info
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types

import transformers
transformers.logging.set_verbosity_error()  # hide harmless "unexpected key" load warnings

pd.set_option("display.max_colwidth", 120)
np.random.seed(0)

/home/saksham/.pyenv/versions/3.12.0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---- Configuration (everything tunable lives here) ----
DATA_DIR   = Path("data")
PDF_PATH   = DATA_DIR / "catan_rules.pdf"
PDF_URL    = "https://www.catan.com/sites/default/files/2021-06/catan_base_rules_2020_200707.pdf"

BI_ENCODER    = "sentence-transformers/all-MiniLM-L6-v2"   # dense embedding model (Bi-Encoder)
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker (Cross-Encoder)
# Tried in order. On the free tier, models were often overloaded (HTTP 503) or hit the
# 20-requests/day/model quota (HTTP 429), so the notebook falls back down this list.
GEMINI_MODELS = ["gemini-3.6-flash", "gemini-3.5-flash", "gemini-3.8-flash", "gemini-3.7-flash",
                 "gemini-flash-lite-latest", "gemini-3.5-flash-lite"]

RETRIEVE_K = 10      # candidates fetched by the bi-encoder
CONTEXT_K  = 3       # chunks passed to the LLM
CHUNK_WORDS = 90     # target chunk size (words)
OVERLAP_SENTS = 1    # sentences repeated between consecutive chunks

# The API key is read from the environment or a local .env file (never hard-coded / committed)
load_dotenv(override=True)   # .env wins over any stale key exported in the shell
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    print("No GEMINI_API_KEY found: answers will be read from llm_cache.json only")

No GEMINI_API_KEY found: answers will be read from llm_cache.json only


## 2. Load and clean the PDF

The rulebook uses a two-column layout with decorative elements, so plain `get_text()` mixes columns and picks up noise.
The extractor below:
* reads text **block by block**, then orders blocks left column first, then right column, top to bottom;
* drops spans that are **not body text**: font size < 6 pt (text printed inside card artwork), the decorative *Ogham* border strip at the foot of each page, and the © lines embedded in illustrations;
* removes the Almanac cross-reference marker (a sun glyph that extracts as a stray `Y`), index dot-leaders, hyphenated line breaks and illustration labels.

Page 1 (cover) is skipped.

In [3]:
# Download the rulebook once (it is not committed to the repo)
DATA_DIR.mkdir(exist_ok=True)
if not PDF_PATH.exists():
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    PDF_PATH.write_bytes(urllib.request.urlopen(req).read())
doc = fitz.open(PDF_PATH)
print(f"{PDF_PATH} — {len(doc)} pages")

data/catan_rules.pdf — 16 pages


In [4]:
def page_text(page):
    # Extract body text from one page in reading order (left column, then right column).
    mid = page.rect.width / 2
    blocks = []
    for b in page.get_text("dict")["blocks"]:
        lines = []
        for line in b.get("lines", []):
            # keep only real body-text spans
            t = "".join(s["text"] for s in line["spans"]
                        if s["size"] >= 6 and "Ogham" not in s["font"] and "©" not in s["text"])
            if t.strip():
                lines.append(t)
        if lines:
            x0, y0 = b["bbox"][:2]
            column = 0 if x0 < mid - 20 else 1
            blocks.append((column, y0, "\n".join(lines)))
    blocks.sort()                                   # column first, then vertical position
    return "\n".join(text for _, _, text in blocks)


def clean(text):
    # Normalise extracted text: drop index lines, fix hyphenation, remove layout artefacts.
    text = "\n".join(l for l in text.split("\n") if not re.search(r"(\.\s+){4,}", l))  # index dot leaders
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)          # "rel-\nevant" -> "relevant"
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+Y(?=[\s.,;:)”\"!?]|$)", "", text)  # Almanac cross-reference glyph
    text = re.sub(r"Illustration [A-Z]( [A-Z0-9]{1,2})*(?= [A-Z][a-z]| \(|$)", "", text)  # figure labels
    return re.sub(r"\s+", " ", text).strip()


pages = [(i + 1, clean(page_text(doc[i]))) for i in range(1, len(doc))]   # skip the cover
print(sum(len(t.split()) for _, t in pages), "words of body text")
print("\nSample (page 5):\n", textwrap.fill(pages[3][1][:600], 110))

8060 words of body text

Sample (page 5):
 b) Settlement Requires: Brick, Lumber, Wool, & Grain Take special note of the “Distance Rule”: you may only
build a settlement at an intersection if all 3 of the adjacent intersections are vacant (i.e., none are
occupied by any settlements or cities—even yours). Each of your settlements must connect to at least 1 of your
own roads. Regardless of whose turn it is (i.e., during any production phase), when a terrain hex produces
resources, you receive 1 resource card for each settlement you have adjacent to that terrain hex. Each
settlement is worth 1 victory point. c) City Requires: 3 Ore & 2 Gr


## 3. Chunking

Sentence-aware fixed-size chunking: sentences are packed into a chunk until it reaches ~90 words, and the last sentence is
repeated at the start of the next chunk (1-sentence overlap) so a rule split across a boundary is not lost.
Chunks never cross page boundaries, and every chunk keeps its page number for traceability.

Both pipelines use **this exact same list of chunks**.

In [5]:
def chunk_pages(pages, max_words=CHUNK_WORDS, overlap_sents=OVERLAP_SENTS):
    chunks = []
    for page_no, text in pages:
        # split on sentence ends that are followed by a capital letter / bullet / quote
        sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z“(•])", text)
        current = []
        for s in sentences:
            current.append(s)
            if sum(len(x.split()) for x in current) >= max_words:
                chunks.append({"page": page_no, "text": " ".join(current)})
                current = current[-overlap_sents:] if overlap_sents else []
        # flush the remainder of the page (skip tiny leftovers that are only overlap)
        if current and sum(len(x.split()) for x in current) > 12:
            chunks.append({"page": page_no, "text": " ".join(current)})
    for i, c in enumerate(chunks):
        c["id"] = i
    return chunks


chunks = chunk_pages(pages)
lengths = [len(c["text"].split()) for c in chunks]
print(f"{len(chunks)} chunks | words per chunk: min {min(lengths)}, median {int(np.median(lengths))}, max {max(lengths)}")
pd.DataFrame(chunks)[["id", "page", "text"]].head(8)

106 chunks | words per chunk: min 13, median 96, max 191


,id,page,text
0,0,2,"Dear Settlers, To make it as easy as possible for you to start playing Catan®, we use an award-winning rules system,..."
1,1,2,Have fun settling this new land together! — Klaus Teuber You can find further information at: catan.com catanstudio....
2,2,2,Game Components • 19 terrain hexes (tiles) • 6 sea frame pieces • 9 harbor pieces • 18 circular number tokens • 95 r...
3,3,2,"Before building the island, assemble the frame by matching the numbers at the ends of the frame pieces together (i.e..."
4,4,2,"Before your first game, you must remove the die-cut components from the cardboard holders. Carefully punch out and s..."
5,5,2,"Third, place the circular number tokens on top of the designated terrain hexes. Finally, place your settlements and ..."
6,6,3,"Starting Map for Beginners To make it as easy as possible for you to get started with Catan, we use an award- winnin..."
7,7,3,"And finally, if you have questions during the game, please consult the Almanac (it begins on page 6). 9 H 11 11 8 H ..."


## 4. Pipeline 1 — Baseline retrieval (Bi-Encoder)

The bi-encoder embeds the **query and each chunk independently** into 384-d vectors. Embeddings are L2-normalised,
so the inner product equals cosine similarity. The "vector store" is a NumPy matrix with exact (brute-force) search,
which is fine for ~100 chunks; a library like FAISS or Chroma would replace `search()` at larger scale.

In [6]:
bi_encoder = SentenceTransformer(BI_ENCODER)


class VectorStore:
    # Minimal dense vector store: normalised embeddings + exact inner-product search.

    def __init__(self, model, chunks):
        self.model, self.chunks = model, chunks
        t0 = time.perf_counter()
        self.embeddings = model.encode([c["text"] for c in chunks],
                                       normalize_embeddings=True, batch_size=32)
        self.index_seconds = time.perf_counter() - t0

    def search(self, query, k=RETRIEVE_K):
        q = self.model.encode([query], normalize_embeddings=True)[0]
        scores = self.embeddings @ q                    # cosine similarity with every chunk
        top = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in top]


store = VectorStore(bi_encoder, chunks)
print(f"Indexed {len(chunks)} chunks -> matrix {store.embeddings.shape} in {store.index_seconds:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8753.89it/s]

Indexed 106 chunks -> matrix (106, 384) in 1.43s


In [7]:
def baseline_retrieve(query):
    # Pipeline 1: Top-10 by cosine similarity; the first 3 go to the LLM.
    top10 = store.search(query, RETRIEVE_K)
    return top10, top10[:CONTEXT_K]

## 5. Pipeline 2 — Re-ranking (Cross-Encoder)

The cross-encoder reads the **query and a chunk together** as one input (`[CLS] query [SEP] chunk`), so every query token
can attend to every chunk token. It outputs a single relevance score per pair. We score the same Top-10 candidates
from Pipeline 1, sort them by that score and keep the new Top-3.

In [8]:
cross_encoder = CrossEncoder(CROSS_ENCODER)


def rerank(query, candidates):
    # Score every (query, chunk) pair jointly and sort descending by cross-encoder score.
    pairs = [(query, chunks[i]["text"]) for i, _ in candidates]
    scores = cross_encoder.predict(pairs)
    order = np.argsort(-scores)
    return [(candidates[j][0], float(scores[j])) for j in order]


def rerank_retrieve(query):
    # Pipeline 2: baseline Top-10 -> cross-encoder re-rank -> Top-3.
    top10, _ = baseline_retrieve(query)
    reranked = rerank(query, top10)
    return top10, reranked, reranked[:CONTEXT_K]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6151.12it/s]

### LLM answer generation (identical prompt for both pipelines)

Only the three context chunks differ between Answer 1 and Answer 2. Temperature is 0 so differences come from the
context, not from sampling.

In [9]:
client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

PROMPT_TEMPLATE = (
    "You are a rules assistant for the board game Catan.\n"
    "Answer the question using ONLY the rulebook excerpts below. "
    "If the excerpts do not contain the rule needed, say so instead of guessing.\n"
    "Answer in 2-4 sentences and cite excerpts like [1].\n\n"
    "{context}\n\n"
    "Question: {question}\nAnswer:"
)


def build_prompt(question, context_hits):
    context = "\n\n".join(f"[{n}] (page {chunks[i]['page']}) {chunks[i]['text']}"
                          for n, (i, _) in enumerate(context_hits, start=1))
    return PROMPT_TEMPLATE.format(context=context, question=question)


FALLBACK_CODES = (429, 500, 503)   # quota used up / transient overload -> try the next model

# Every Gemini answer is cached on disk (keyed by model + prompt), so re-running the notebook
# does not spend free-tier quota (20 requests per model per day) and the raw answers are kept.
CACHE_PATH = Path("llm_cache.json")
llm_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}


def cache_key(prompt, model):
    return f"{model}::{hashlib.sha256(prompt.encode()).hexdigest()}"


def generate(prompt, model, retries=3):
    # Call one Gemini model (or return its cached answer). Retries only on 500/503 overloads.
    key = cache_key(prompt, model)
    if key in llm_cache:
        return llm_cache[key]["text"], llm_cache[key]["seconds"]
    if client is None:
        raise RuntimeError("answer not in llm_cache.json and no GEMINI_API_KEY set")
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            resp = client.models.generate_content(
                model=model, contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0))
            text, seconds = resp.text.strip(), time.perf_counter() - t0
            llm_cache[key] = {"model": model, "text": text, "seconds": seconds}
            CACHE_PATH.write_text(json.dumps(llm_cache, indent=1, ensure_ascii=False))
            return text, seconds
        except genai.errors.APIError as e:
            # 429 = daily quota for this model is exhausted; waiting will not help, so fail fast
            if e.code in (500, 503) and attempt < retries - 1:
                time.sleep(15 * (attempt + 1))
                continue
            raise


def answer_both(question):
    # Run both pipelines end-to-end for one question.
    # Answer 1 and Answer 2 always come from the SAME model, so the only difference is the context.
    # If a model fails, both answers are regenerated with the next model; a model that works
    # is moved to the front so later questions use it too.
    top10, base_ctx = baseline_retrieve(question)
    _, reranked, rr_ctx = rerank_retrieve(question)
    p1, p2 = build_prompt(question, base_ctx), build_prompt(question, rr_ctx)
    # Re-runs reuse a model that already answered BOTH prompts, so results stay reproducible
    for model in GEMINI_MODELS:
        if cache_key(p1, model) in llm_cache and cache_key(p2, model) in llm_cache:
            (ans1, t1), (ans2, t2) = generate(p1, model), generate(p2, model)
            return dict(top10=top10, reranked=reranked, model=model,
                        answer1=ans1, answer2=ans2, llm_s1=t1, llm_s2=t2)
    for model in list(GEMINI_MODELS):
        try:
            ans1, t1 = generate(p1, model)
            ans2, t2 = generate(p2, model)
            GEMINI_MODELS.remove(model); GEMINI_MODELS.insert(0, model)
            return dict(top10=top10, reranked=reranked, model=model,
                        answer1=ans1, answer2=ans2, llm_s1=t1, llm_s2=t2)
        except genai.errors.APIError as e:
            if e.code not in FALLBACK_CODES:
                raise
            print(f"  {model} unavailable ({e.code}), falling back")
    raise RuntimeError("all configured Gemini models are unavailable — try again later")

## 6. Failure-mode analysis

Each failure query has a **gold chunk**: the chunk containing the sentence that actually decides the answer, identified by
reading the rulebook. It is located automatically by a short phrase from that sentence, so the rank tracking below is not
hand-edited.

| # | Failure type | What makes it hard for the bi-encoder |
|---|---|---|
| Q1 | Keyword trap / distractor | Many chunks mention "ore", "wool", "harbor", "3:1"; the deciding sentence is a negation |
| Q2 | Negation | "never gets a number token" sits in the variable set-up section, not in the Desert entry |
| Q3 | Constraint (limit) | "1 knight card **or** 1 progress card" — the limit is phrased differently from the question |
| Q4 | Corner case buried at rank 8 | Generic "how to build" chunks crowd out the one note about running out of settlements |

These four were selected from the wider pool in Section 7 as the cases where the gold chunk sat at ranks 4–10.
Section 7 also reports the cases where re-ranking did *not* help.

In [10]:
FAILURE_QUERIES = [
    ("Q1", "Keyword trap",
     "If I have a settlement on the ore 2:1 harbor, can I trade wool at 3:1?",
     "does not permit you to trade any other resource"),
    ("Q2", "Negation",
     "Which hex never receives a number token?",
     "desert never gets a number token"),
    ("Q3", "Constraint",
     "Can I play a knight and a progress card in the same turn?",
     "either 1 knight card or 1 progress card"),
    ("Q4", "Corner case",
     "I have built all 5 settlements. How can I build another one?",
     "must upgrade 1 of your settlements"),
]


def gold_ids(phrase):
    return [c["id"] for c in chunks if phrase.lower() in c["text"].lower()]


def rank_of(ids, hits):
    # 1-based best rank of any gold chunk in a hit list (None if absent).
    order = [i for i, _ in hits]
    ranks = [order.index(g) + 1 for g in ids if g in order]
    return min(ranks) if ranks else None


for qid, kind, q, phrase in FAILURE_QUERIES:
    print(qid, "gold chunk(s):", gold_ids(phrase))

Q1 gold chunk(s): [57]
Q2 gold chunk(s): [83]
Q3 gold chunk(s): [39]
Q4 gold chunk(s): [76]


In [11]:
# Run both pipelines (including Gemini) for every failure query
results = {}
for qid, kind, q, phrase in FAILURE_QUERIES:
    results[qid] = answer_both(q)
    time.sleep(2)   # stay well inside free-tier rate limits
print("done")

done


### 6.1 Rank-shift table

In [12]:
rows = []
for qid, kind, q, phrase in FAILURE_QUERIES:
    g = gold_ids(phrase)
    r = results[qid]
    b, a = rank_of(g, r["top10"]), rank_of(g, r["reranked"])
    rows.append({"query": qid, "type": kind, "gold chunk": g[0] if len(g) == 1 else g,
                 "bi-encoder rank": b, "cross-encoder rank": a,
                 "shift": f"#{b} → #{a}",
                 "in baseline Top-3": b is not None and b <= CONTEXT_K,
                 "in re-ranked Top-3": a is not None and a <= CONTEXT_K})
rank_shift = pd.DataFrame(rows)
rank_shift

,query,type,gold chunk,bi-encoder rank,cross-encoder rank,shift,in baseline Top-3,in re-ranked Top-3
0,Q1,Keyword trap,57,4,3,#4 → #3,False,True
1,Q2,Negation,83,6,1,#6 → #1,False,True
2,Q3,Constraint,39,6,1,#6 → #1,False,True
3,Q4,Corner case,76,8,1,#8 → #1,False,True


In [13]:
def show_rankings(qid):
    # Side-by-side Top-10 before and after re-ranking; the gold chunk is marked with ★.
    _, _, q, phrase = next(x for x in FAILURE_QUERIES if x[0] == qid)
    g = set(gold_ids(phrase))
    r = results[qid]
    df = pd.DataFrame({
        "rank": range(1, RETRIEVE_K + 1),
        "baseline chunk": [f"{'★ ' if i in g else ''}{i}" for i, _ in r["top10"]],
        "cosine": [round(s, 3) for _, s in r["top10"]],
        "re-ranked chunk": [f"{'★ ' if i in g else ''}{i}" for i, _ in r["reranked"]],
        "CE score": [round(s, 2) for _, s in r["reranked"]],
        "re-ranked text (start)": [chunks[i]["text"][:90] + "…" for i, _ in r["reranked"]],
    })
    print(f"{qid}: {q}")
    return df

show_rankings("Q1")

Q1: If I have a settlement on the ore 2:1 harbor, can I trade wool at 3:1?


,rank,baseline chunk,cosine,re-ranked chunk,CE score,re-ranked text (start)
0,1,58,0.590,55,4.67,"You do not need a harbor (settlement at a harbor location) to trade at 4:1, so when nobody…"
1,2,102,0.579,56,4.22,There are 2 different kinds of harbor locations: Generic Harbor (3:1): Here you may exchan…
2,3,55,0.579,★ 57,3.46,"So, if you earn plenty of a certain type of resource, it can be useful to build on the spe…"
3,4,★ 57,0.572,102,2.48,However [E] is also at a wool harbor. Since it’s impossible for you to have settlements ad…
4,5,56,0.514,17,2.42,The other players can also make their own proposals and counteroffers. Important: Players …
5,6,42,0.484,101,1.75,"If a “10” is rolled, the red player receives 1 ore card and the orange player receives 1 b…"
6,7,101,0.458,41,0.10,"Coleman, the blue player, wants to build a settlement. The settlements marked “A” are alre…"
7,8,41,0.450,58,-1.79,"He can also trade 4 ore cards for any 2 other cards. If he traded 4 wool instead of 4 ore,…"
8,9,88,0.450,42,-2.09,"However, you cannot give away cards, or trade matching resources (“trade” 3 ore for 1 ore,…"
9,10,17,0.441,88,-9.61,"Trade After you roll for resource production, you may trade with other players (domestic t…"


In [14]:
show_rankings("Q2")

Q2: Which hex never receives a number token?


,rank,baseline chunk,cosine,re-ranked chunk,CE score,re-ranked text (start)
0,1,59,0.670,★ 83,5.04,"Important: Alternatively, you can use a fully random set-up. Place 1 token on each land he…"
1,2,100,0.440,15,1.87,Each player who has a settlement on an intersection that borders a terrain hex marked with…
2,3,5,0.431,100,1.52,"If, for example, a “10” is rolled, all terrain hexes with a “10” number token produce reso…"
3,4,105,0.411,59,1.23,N Number Tokens 8 ••••• ••••• The 18 number tokens are marked with the numerals “2” throug…
4,5,60,0.408,99,0.47,"Each city is worth 2 victory points. To build or upgrade, you need to acquire resources. H…"
5,6,★ 83,0.400,5,-0.74,"Third, place the circular number tokens on top of the designated terrain hexes. Finally, p…"
6,7,64,0.397,64,-1.08,You receive your income for each terrain hex adjacent to your settlements or cities every …
7,8,104,0.369,60,-1.78,“6” and “8” (the red numbers) are the most frequently rolled numbers. They each have 5 dot…
8,9,15,0.347,104,-4.14,The red numbers 6 and 8 are the tallest numbers with the most pips; they are likely to be …
9,10,99,0.342,105,-6.84,"However, these same high-producing hexes are often the primary target for the robber. 6 7 …"


In [15]:
show_rankings("Q3")

Q3: Can I play a knight and a progress card in the same turn?


,rank,baseline chunk,cosine,re-ranked chunk,CE score,re-ranked text (start)
0,1,28,0.538,★ 39,9.50,This keeps the other players in the dark. You cannot trade or give away development cards.…
1,2,61,0.517,28,5.24,If another player has more knight cards in front of them than the current holder of the La…
2,3,27,0.502,26,5.11,"Important: If the production number for the hex containing the robber is rolled, the owner…"
3,4,49,0.476,61,4.37,Only one road can be built on any path. At each end of a path is an intersection. Progress…
4,5,50,0.398,47,3.88,The influence (for resource yields) of settlements and cities extends into the 3 adjacent …
5,6,★ 39,0.398,62,2.01,"You may use these cards to build in the same turn. Monopoly: If you play this card, you mu…"
6,7,84,0.375,50,1.55,"On Niall’s turn, he plays a knight card and moves the robber from the fields hex to the hi…"
7,8,62,0.353,49,1.09,They must answer truthfully.) If you are the first player to have 3 knight cards face up i…
8,9,26,0.350,27,0.89,"Knight Cards (Purple Frame) K If you play a knight card, you must immediately move the rob…"
9,10,47,0.340,84,-3.67,More set-up instructions can be found in “Set-up Phase.” Soldier Cards Some earlier editio…


In [16]:
show_rankings("Q4")

Q4: I have built all 5 settlements. How can I build another one?


,rank,baseline chunk,cosine,re-ranked chunk,CE score,re-ranked text (start)
0,1,19,0.624,★ 76,7.84,"Example: See Illustration M. Olivia, the blue player, wants to build a new settlement. She…"
1,2,86,0.607,34,3.77,C Cities You cannot build a city directly. You can only upgrade an existing settlement to …
2,3,32,0.604,32,3.35,"To build, you must turn in the specified combinations of resource cards (see the Building …"
3,4,22,0.593,19,3.21,"To build, you must pay specific combinations of resource cards (see the Building Costs Car…"
4,5,21,0.580,103,3.13,You can only build a new settlement on an unoccupied intersection if you have a road leadi…
5,6,34,0.564,21,2.50,"b) Settlement Requires: Brick, Lumber, Wool, & Grain Take special note of the “Distance Ru…"
6,7,8,0.547,86,-0.01,"There are, nevertheless, some common points you should consider: • Brick and lumber are th…"
7,8,★ 76,0.533,22,-1.82,Each settlement is worth 1 victory point. c) City Requires: 3 Ore & 2 Grain = You may only…
8,9,103,0.516,80,-2.54,The second road must attach to the second settlement (pointing in any of the 3 directions)…
9,10,80,0.516,8,-2.70,"The board changes each game. If you would like to use the variable set-up, you can find th…"


### 6.2 Answer 1 (baseline context) vs. Answer 2 (re-ranked context)

In [17]:
for qid, kind, q, phrase in FAILURE_QUERIES:
    r = results[qid]
    g = gold_ids(phrase)
    print("=" * 110)
    print(f"{qid} [{kind}] {q}   (model: {r['model']})")
    print(f"Gold chunk {g} — decisive text: “…{phrase}…”")
    print(f"\nBaseline context chunks: {[i for i, _ in r['top10'][:3]]}")
    print(textwrap.fill("ANSWER 1: " + r["answer1"], 110))
    print(f"\nRe-ranked context chunks: {[i for i, _ in r['reranked'][:3]]}")
    print(textwrap.fill("ANSWER 2: " + r["answer2"], 110))
print("=" * 110)

Q1 [Keyword trap] If I have a settlement on the ore 2:1 harbor, can I trade wool at 3:1?   (model: gemini-flash-lite-latest)
Gold chunk [57] — decisive text: “…does not permit you to trade any other resource…”

Baseline context chunks: [58, 102, 55]
ANSWER 1: Based on the provided excerpts, there is no mention of an "ore 2:1 harbor" or the specific rules for
trading wool at 3:1. The rulebook does state that at a generic harbor, you may exchange 3 identical resource
cards for any 1 other resource card [3].

Re-ranked context chunks: [55, 56, 57]
ANSWER 2: No, a special harbor does not permit you to trade any other resource type at a more favorable rate,
not even 3:1 [3]. The exchange rate of 2:1 only applies to the resource shown on the harbor location [3].
Q2 [Negation] Which hex never receives a number token?   (model: gemini-flash-lite-latest)
Gold chunk [83] — decisive text: “…desert never gets a number token…”

Baseline context chunks: [59, 100, 5]
ANSWER 1: Based on the provided r

In [18]:
# Full text of each gold chunk, for checking the answers against the rulebook
for qid, kind, q, phrase in FAILURE_QUERIES:
    for g in gold_ids(phrase):
        print(f"{qid} gold chunk {g} (page {chunks[g]['page']}):\n{textwrap.fill(chunks[g]['text'], 110)}\n")

Q1 gold chunk 57 (page 9):
So, if you earn plenty of a certain type of resource, it can be useful to build on the special harbor for that
resource type. The exchange rate of 2:1 only applies to the resource shown on the harbor location. A special
harbor does not permit you to trade any other resource type at a more favorable rate (not even 3:1)! Example:
Ron built a settlement at the ore special harbor. He may exchange 2 ore cards for any 1 other resource card.
He can also trade 4 ore cards for any 2 other cards.

Q2 gold chunk 83 (page 13):
Important: Alternatively, you can use a fully random set-up. Place 1 token on each land hex. Start at one
corner of the island, and place the number tokens in random order. In such case, the tokens with the red
numbers must not be next to each other. You may have to swap tokens to ensure that no red numbers are on
adjacent hexes. Note: The desert never gets a number token. It should be skipped. More set-up instructions can
be found in “Set-up Phase

## 7. Wider evaluation over a pool of rule questions

To check that the failure cases are not cherry-picked in isolation, the same rank comparison is run over a larger pool of
29 rule questions (each with its gold phrase). This is retrieval-only (no LLM calls).
It reports where re-ranking **helps**, where it **hurts**, and where it **cannot help** because the gold chunk never made
the bi-encoder's Top-10 (re-ranking can only reorder what stage 1 retrieved).

In [19]:
QUERY_POOL = [
 ("exception", "Can I play a development card on the same turn that I bought it?", "bought during the same turn"),
 ("exception", "What happens if the bank runs out of a resource and only one player is owed it?", "only affects a single player"),
 ("exception", "I hold 9 resource cards when someone rolls a 7. How many must I discard?", "if you have 9 resource cards, you discard 4"),
 ("exception", "My longest road was cut by an opponent's settlement and now I am tied with another player. Do I keep the Longest Road card?", "you still keep the"),
 ("exception", "Can I use a harbor on the same turn I build a settlement there?", "use a harbor on the same turn"),
 ("exception", "I have built all 5 settlements. How can I build another one?", "must upgrade 1 of your settlements"),
 ("negation", "Can two players trade with each other when it is not either of their turns?", "may not trade among themselves"),
 ("negation", "Is it allowed to leave the robber on the hex where it already is?", "may not choose to leave the"),
 ("negation", "Am I allowed to give resource cards to another player for free?", "cannot give away cards"),
 ("negation", "Is it prohibited to trade development cards?", "cannot trade or give away development cards"),
 ("negation", "Can I win the game during another player's turn?", "You can only win during your turn"),
 ("negation", "Am I allowed to trade with the bank when it is not my turn?", "may not trade with the bank during another"),
 ("negation", "Which hex never receives a number token?", "desert never gets a number token"),
 ("negation", "Can I build a settlement right next to one of my own settlements?", "even yours"),
 ("negation", "Can I trade 2 wool for 1 wool?", "trade like resources"),
 ("keyword", "If I have a settlement on the ore 2:1 harbor, can I trade wool at 3:1?", "does not permit you to trade any other resource"),
 ("keyword", "How many pieces of each building type does a player get?", "15 roads, 5 settlements, and 4"),
 ("keyword", "What does the Monopoly card do?", "name 1 type of resource"),
 ("keyword", "What happens when I rob a player who has no resource cards?", "If that player has no cards"),
 ("keyword", "Does a branching road count all of its branches toward Longest Road?", "single longest branch"),
 ("keyword", "What happens if nobody has a road of 5 or more segments after a road is broken?", "Do the same if no one has a 5+"),
 ("keyword", "In the second round of set-up, which direction do players go?", "counterclockwise"),
 ("keyword", "When do I get my starting resources in the set-up phase?", "immediately after building their second settlement"),
 ("keyword", "Can red number tokens be placed next to each other?", "red numbers must not be next"),
 ("keyword", "Do roads give victory points?", "Roads provide victory points only if"),
 ("keyword", "Can I play a knight card before rolling the dice?", "even before you roll the dice"),
 ("keyword", "Can I play a knight and a progress card in the same turn?", "either 1 knight card or 1 progress card"),
 ("keyword", "How many victory points do I need to win?", "10 victory points"),
 ("keyword", "Can I win with victory point cards I just bought?", "may immediately reveal this card"),
]

pool_rows = []
for kind, q, phrase in QUERY_POOL:
    g = gold_ids(phrase)
    assert g, f"gold phrase not found: {phrase}"
    top10, reranked, _ = rerank_retrieve(q)
    b, a = rank_of(g, top10), rank_of(g, reranked)
    if b is None:
        outcome = "not in Top-10 (can't fix)"
    elif a < b:
        outcome = "improved"
    elif a > b:
        outcome = "worse"
    else:
        outcome = "unchanged"
    pool_rows.append({"type": kind, "question": q, "bi rank": b, "CE rank": a, "outcome": outcome})

pool = pd.DataFrame(pool_rows)
pool

,type,question,bi rank,CE rank,outcome
0,exception,Can I play a development card on the same turn that I bought it?,1.0,1.0,unchanged
1,exception,What happens if the bank runs out of a resource and only one player is owed it?,1.0,1.0,unchanged
2,exception,I hold 9 resource cards when someone rolls a 7. How many must I discard?,1.0,1.0,unchanged
3,exception,My longest road was cut by an opponent's settlement and now I am tied with another player. Do I keep the Longest Roa...,2.0,1.0,improved
4,exception,Can I use a harbor on the same turn I build a settlement there?,NaN,NaN,not in Top-10 (can't fix)
5,exception,I have built all 5 settlements. How can I build another one?,8.0,1.0,improved
6,negation,Can two players trade with each other when it is not either of their turns?,2.0,2.0,unchanged
7,negation,Is it allowed to leave the robber on the hex where it already is?,3.0,1.0,improved
8,negation,Am I allowed to give resource cards to another player for free?,4.0,6.0,worse
9,negation,Is it prohibited to trade development cards?,1.0,1.0,unchanged


In [20]:
def hit_at(ranks, k):
    return np.mean([r is not None and r <= k for r in ranks])

def mrr(ranks):
    # Mean reciprocal rank within the Top-10 (0 if the gold chunk was not retrieved).
    return np.mean([1 / r if r else 0 for r in ranks])

b_ranks = [None if pd.isna(r) else int(r) for r in pool["bi rank"]]
a_ranks = [None if pd.isna(r) else int(r) for r in pool["CE rank"]]
summary = pd.DataFrame({
    "metric": ["Hit@1", "Hit@3 (what the LLM sees)", "Hit@10", "MRR@10"],
    "Pipeline 1 (bi-encoder)": [hit_at(b_ranks, 1), hit_at(b_ranks, 3), hit_at(b_ranks, 10), mrr(b_ranks)],
    "Pipeline 2 (re-ranked)":  [hit_at(a_ranks, 1), hit_at(a_ranks, 3), hit_at(a_ranks, 10), mrr(a_ranks)],
}).round(3)
display(summary)
display(pool["outcome"].value_counts().rename("count").to_frame())

,metric,Pipeline 1 (bi-encoder),Pipeline 2 (re-ranked)
0,Hit@1,0.345,0.655
1,Hit@3 (what the LLM sees),0.655,0.828
2,Hit@10,0.862,0.862
3,MRR@10,0.533,0.741


,count
outcome,
unchanged,13
improved,11
not in Top-10 (can't fix),4
worse,1


## 8. Latency profiling

Each stage is timed separately with `time.perf_counter()` after a warm-up call, repeated over all 4 failure queries × 20 runs.
* **Pipeline 1 retrieval** = embed the query + similarity search over all chunks (Top-10)
* **Pipeline 2 retrieval** = Pipeline 1 retrieval + cross-encoder scoring of 10 pairs + sort
* **LLM generation** = the Gemini call measured in Section 6 (network + generation; same prompt shape for both pipelines)

Hardware: CPU only.

In [21]:
import platform, torch
print(platform.processor() or platform.machine(), "| torch threads:", torch.get_num_threads(), "| CUDA:", torch.cuda.is_available())

REPEATS = 20
queries = [q for _, _, q, _ in FAILURE_QUERIES]

# warm-up (first calls include lazy initialisation)
baseline_retrieve(queries[0]); rerank_retrieve(queries[0])

def time_ms(fn, q):
    t0 = time.perf_counter(); fn(q); return (time.perf_counter() - t0) * 1000

t_base, t_rr, t_ce_only = [], [], []
for _ in range(REPEATS):
    for q in queries:
        t_base.append(time_ms(baseline_retrieve, q))
        t_rr.append(time_ms(rerank_retrieve, q))
        top10 = store.search(q)
        t_ce_only.append(time_ms(lambda x: rerank(x, top10), q))

llm1 = [results[k]["llm_s1"] * 1000 for k in results]
llm2 = [results[k]["llm_s2"] * 1000 for k in results]

def stats(xs):
    xs = sorted(xs)
    return {"mean (ms)": statistics.mean(xs), "median (ms)": statistics.median(xs),
            # a 95th percentile is meaningless for a handful of samples (e.g. the 4 LLM calls)
            "p95 (ms)": xs[int(0.95 * (len(xs) - 1))] if len(xs) >= 20 else float("nan"), "n": len(xs)}

latency = pd.DataFrame({
    "Pipeline 1 retrieval (embed + top-10)": stats(t_base),
    "Cross-encoder re-rank only (10 pairs)": stats(t_ce_only),
    "Pipeline 2 retrieval (embed + top-10 + re-rank)": stats(t_rr),
    "LLM generation — Answer 1": stats(llm1),
    "LLM generation — Answer 2": stats(llm2),
}).T.round(2)
latency["n"] = latency["n"].astype(int)
latency

x86_64 | torch threads: 16 | CUDA: False


,mean (ms),median (ms),p95 (ms),n
Pipeline 1 retrieval (embed + top-10),14.59,13.92,24.02,80
Cross-encoder re-rank only (10 pairs),168.62,136.21,272.32,80
Pipeline 2 retrieval (embed + top-10 + re-rank),201.06,184.07,295.36,80
LLM generation — Answer 1,1196.73,1085.68,NaN,4
LLM generation — Answer 2,1039.18,1022.29,NaN,4


In [22]:
end_to_end = pd.DataFrame({
    "retrieval (ms)": [latency.loc["Pipeline 1 retrieval (embed + top-10)", "median (ms)"],
                       latency.loc["Pipeline 2 retrieval (embed + top-10 + re-rank)", "median (ms)"]],
    "LLM (ms)": [latency.loc["LLM generation — Answer 1", "median (ms)"],
                 latency.loc["LLM generation — Answer 2", "median (ms)"]],
}, index=["Pipeline 1", "Pipeline 2"])
end_to_end["total (ms)"] = end_to_end.sum(axis=1)
end_to_end["retrieval overhead vs P1"] = (end_to_end["retrieval (ms)"] / end_to_end["retrieval (ms)"].iloc[0]).map("{:.1f}x".format)
end_to_end.round(1)

,retrieval (ms),LLM (ms),total (ms),retrieval overhead vs P1
Pipeline 1,13.9,1085.7,1099.6,1.0x
Pipeline 2,184.1,1022.3,1206.4,13.2x


## 9. Scaling measurement

Measurements to support the scaling question: how the cost of cross-encoding grows if it were applied to **every** chunk
instead of only the Top-10, compared with bi-encoder search where chunk embeddings are computed once, offline.

In [23]:
q = queries[0]
all_pairs = [(q, c["text"]) for c in chunks]

t0 = time.perf_counter(); cross_encoder.predict(all_pairs, batch_size=32); ce_all_s = time.perf_counter() - t0
ce_per_pair_ms = ce_all_s * 1000 / len(chunks)

t0 = time.perf_counter()
for _ in range(50): store.search(q)
search_ms = (time.perf_counter() - t0) * 1000 / 50

# one-off cost of embedding the corpus for the bi-encoder (done at indexing time, not per query)
embed_per_chunk_ms = store.index_seconds * 1000 / len(chunks)

scale = pd.DataFrame({"corpus size (chunks)": [len(chunks), 10_000, 1_000_000]})
scale["CE over all chunks, per query (s)"] = scale["corpus size (chunks)"] * ce_per_pair_ms / 1000
scale["CE over Top-10 only, per query (s)"] = RETRIEVE_K * ce_per_pair_ms / 1000
scale["bi-encoder one-off indexing (s)"] = scale["corpus size (chunks)"] * embed_per_chunk_ms / 1000
print(f"Measured on this corpus: CE {ce_per_pair_ms:.2f} ms/pair | brute-force vector search {search_ms:.2f} ms/query "
      f"| embedding {embed_per_chunk_ms:.2f} ms/chunk")
print("Rows for 10k and 1M chunks are linear extrapolations of the measured per-pair cost (CPU).")
scale.round(3)

Measured on this corpus: CE 20.23 ms/pair | brute-force vector search 16.77 ms/query | embedding 13.46 ms/chunk
Rows for 10k and 1M chunks are linear extrapolations of the measured per-pair cost (CPU).


,corpus size (chunks),"CE over all chunks, per query (s)","CE over Top-10 only, per query (s)",bi-encoder one-off indexing (s)
0,106,2.144,0.202,1.427
1,10000,202.293,0.202,134.630
2,1000000,20229.295,0.202,13462.981


---
*Report sections (architecture comparison, rank-shift discussion, latency and scaling answers) are written separately in
the report, based on the outputs above.*